# GridapTensorProduct.jl — Library Walk-Through

This notebook traces the **five-stage pipeline** of GridapTensorProduct.jl from raw 1D
meshes all the way to a solved linear system, explaining how each structure is built and
how the stages connect.

**Domain:** 2D Poisson on Ω = [0,1]² with homogeneous Dirichlet BCs and unit forcing:
$$-\Delta u = 1 \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

---
## Architecture overview

```
Stage 1  TensorProductMeasure        — bundle N subdomain quadrature measures
Stage 2  TensorProductFESpace        — bundle N subdomain FE spaces + DOF mapping
Stage 3  TensorProductGeometry       — triangulation and cell-field wrappers
Stage 4  TensorProductFEMOperators   — extract 1D matrices; form Kronecker global ops
Stage 5  TensorProductAssembly       — high-level API; assemble and return (A, b)
```

Key idea: **work with 1D subdomains independently in Stages 1–4; couple via Kronecker products only at assembly (Stage 5).**

In [ ]:
using Gridap
using GridapTensorProduct
using LinearAlgebra

---
## Stage 1 — 1D meshes and subdomain measures

Everything starts with ordinary Gridap objects on **1D subdomains**. No tensor-product types yet.

In [ ]:
Nx, Ny = 8, 8
model_x = CartesianDiscreteModel((0.0, 1.0), Nx)
model_y = CartesianDiscreteModel((0.0, 1.0), Ny)
Ωx = Interior(model_x)
Ωy = Interior(model_y)

quad_order = 2
dΩx = Measure(Ωx, quad_order)
dΩy = Measure(Ωy, quad_order)

println("Subdomain x: ", num_cells(Ωx), " cells")
println("Subdomain y: ", num_cells(Ωy), " cells")

### TensorProductMeasure — the ⊗ operator

Writing `dΩx ⊗ dΩy` creates a `TensorProductMeasure` that bundles the two measures.
When a user writes `∫(expr) * (dΩx ⊗ dΩy)`, the library intercepts it and creates
a `TensorProductIntegrand` marker — no actual integration happens at this point.
Assembly is deferred to Stage 5.

In [ ]:
dΩ_tp = dΩx ⊗ dΩy

println(typeof(dΩ_tp))           # TensorProductMeasure
println(num_subdomains(dΩ_tp))   # 2

# Intercepted multiplication creates a deferred marker
marker = ∫(x -> 1.0) * dΩ_tp
println(typeof(marker))           # TensorProductIntegrand

---
## Stage 2 — Subdomain FE spaces and the tensor DOF map

Each subdomain gets its own standard Gridap `TestFESpace` / `TrialFESpace`.
Dirichlet BCs are applied **per subdomain** before any tensor coupling.

**DOF ordering convention** (subdomain 1 = fastest-varying):
$$\text{global\_dof}(i_1, i_2) = i_1 + (i_2 - 1) \cdot n_{f,1}$$

This matches the Kronecker convention `kron(M_2, M_1)` where M_1 is innermost.

In [ ]:
reffe = ReferenceFE(lagrangian, Float64, 1)

Vx = TestFESpace(Ωx, reffe; conformity=:H1, dirichlet_tags="boundary")
Vy = TestFESpace(Ωy, reffe; conformity=:H1, dirichlet_tags="boundary")
Ux = TrialFESpace(Vx, 0.0)
Uy = TrialFESpace(Vy, 0.0)

println("Free DOFs on Ωx: ", num_free_dofs(Vx))   # Nx - 1 = 7
println("Free DOFs on Ωy: ", num_free_dofs(Vy))   # Ny - 1 = 7

In [ ]:
V_tp = TensorProductFESpace(Vx, Vy)
U_tp = TensorProductFESpace(Ux, Uy)

n_tp = num_free_dofs(V_tp)   # = num_free_dofs(Vx) * num_free_dofs(Vy)
println("Free DOFs: ", num_free_dofs(Vx), " × ", num_free_dofs(Vy), " = ", n_tp)

# Satisfies Gridap's FESpace interface
println(get_free_dof_ids(V_tp))   # Base.OneTo(49)

---
## Stage 3 — Tensor-product geometry

`TensorProductTriangulation` wraps N component triangulations.
Its `num_cells` equals the product of component cell counts.
Access individual components with integer indexing.

In [ ]:
trian_tp = get_triangulation(V_tp)

println(typeof(trian_tp))                             # TensorProductTriangulation
println("Total tensor cells: ", num_cells(trian_tp))  # 8 * 8 = 64
println("Cells on Ωx: ",        num_cells(trian_tp[1]))
println("Cells on Ωy: ",        num_cells(trian_tp[2]))

# Plural accessor for direct tuple access
trians = get_triangulations(V_tp)
println("Number of components: ", length(trians))

---
## Stage 4 — Subdomain operator extraction and Kronecker assembly

### 4a: Assemble fundamental 1D matrices

For each subdomain $k$, `assemble_subdomain_operators` calls Gridap's
`AffineFEOperator` to extract five matrices:

| Matrix | Weak form |
|--------|----------|
| $M^{(k)}$ | $\int_{\Omega_k} \varphi_i \varphi_j \, dx_k$ |
| $K^{(k)}$ | $\int_{\Omega_k} \nabla\varphi_i \cdot \nabla\varphi_j \, dx_k$ |
| $G^{(k)}$ | $\int_{\Omega_k} (\partial_x \varphi_i) \, \varphi_j \, dx_k$ |
| $D^{(k)}$ | $\int_{\Omega_k} \varphi_i \, (\partial_x \varphi_j) \, dx_k$ |
| $A^{(k)}$ | $\int_{\Omega_k} \varphi_i \, (b_k \partial_x \varphi_j) \, dx_k$ |

In [ ]:
ops = assemble_subdomain_operators((Vx, Vy), (dΩx, dΩy))

println(typeof(ops))                         # TensorProductSubdomainOperators{2}
println("M^(x) size: ", size(ops.M_ops[1]))  # (7, 7)
println("K^(x) size: ", size(ops.K_ops[1]))  # (7, 7)
println("M^(y) size: ", size(ops.M_ops[2]))  # (7, 7)
println("K^(y) size: ", size(ops.K_ops[2]))  # (7, 7)

### 4b: Kronecker assembly

Global operators follow the **subdomain 1 = innermost** convention:

| Operator | N=2 formula |
|----------|-------------|
| Mass | `kron(My, Mx)` |
| Stiffness | `kron(My, Kx) + kron(Ky, Mx)` |
| Gradient | `[kron(My, Gx); kron(Gy, Mx)]` (block column) |

In [ ]:
M_global = assemble_mass_tensor(ops)
A_global = assemble_stiffness_tensor(ops)
G_global = assemble_gradient_tensor(ops)

println("Global mass size:      ", size(M_global))  # (49, 49)
println("Global stiffness size: ", size(A_global))  # (49, 49)
println("Global gradient size:  ", size(G_global))  # (98, 49) — 2 blocks stacked
println("Stiffness symmetric:   ", maximum(abs, A_global - A_global') < 1e-14)

# Verify against direct kron
Mx, Kx = ops.M_ops[1], ops.K_ops[1]
My, Ky = ops.M_ops[2], ops.K_ops[2]
A_ref = kron(My, Kx) + kron(Ky, Mx)
println("Max error vs kron reference: ", maximum(abs, A_global - A_ref))

### Low-level API: `TensorProductOperator` (lazy + cached)

In [ ]:
op_stiff = TensorProductOperator(:stiffness, (Vx, Vy), (dΩx, dΩy))

println("Cached before first call: ", op_stiff.global_matrix !== nothing)  # false
A_lazy = get_global_matrix(op_stiff)
println("Cached after first call:  ", op_stiff.global_matrix !== nothing)  # true
println("Same object on repeat:    ", A_lazy === get_global_matrix(op_stiff))

---
## Stage 5 — High-level assembly: `TensorProductAffineOperator`

The recommended entry point for most users. Lazily assembles `(A, b)` on first
access to `get_matrix` / `get_vector` and caches the result.

The bilinear/linear forms passed to the constructor are symbolic documentation;
the actual assembly is driven by `op_type`.

In [ ]:
a(u, v) = ∫(∇(u) ⋅ ∇(v)) * dΩ_tp
l(v)    = ∫(1.0 * v)       * dΩ_tp

op = TensorProductAffineOperator(a, l, U_tp, V_tp;
                                  op_type    = :stiffness,
                                  quad_order = 2)

println("op_type: ",        op.op_type)
println("Cached before: ",  op._matrix !== nothing)

In [ ]:
A = get_matrix(op)
b = get_vector(op)

println("A: ", size(A), "  b: ", length(b))
println("Cached after: ", op._matrix !== nothing)

u_dofs = A \ b
println("Max solution value (≈ 0.073 for -Δu=1): ", round(maximum(u_dofs), digits=4))

### Optional: custom per-subdomain RHS with `rhs_forms`

For separable non-unit forcing, supply per-subdomain linear forms.
The global RHS is assembled as `kron(b_N, ..., kron(b_2, b_1))`.

In [ ]:
# Separable forcing f(x,y) = x * y
f_x(x) = x[1]
f_y(y) = y[1]

op_custom = TensorProductAffineOperator(
    a, l, U_tp, V_tp;
    op_type    = :stiffness,
    quad_order = 2,
    rhs_forms  = (
        (v) -> ∫(f_x * v) * dΩx,
        (v) -> ∫(f_y * v) * dΩy
    )
)
b_custom = get_vector(op_custom)
println("Custom RHS assembled, length: ", length(b_custom))

---
## Extension to 3D: add a third subdomain

The API is **dimension-agnostic**. N=3 requires no code changes beyond adding a third space.

In [ ]:
model_z = CartesianDiscreteModel((0.0, 1.0), 6)
Ωz = Interior(model_z)
dΩz = Measure(Ωz, quad_order)
Vz = TestFESpace(Ωz, reffe; conformity=:H1, dirichlet_tags="boundary")
Uz = TrialFESpace(Vz, 0.0)

V3 = TensorProductFESpace(Vx, Vy, Vz)
U3 = TensorProductFESpace(Ux, Uy, Uz)

op3 = TensorProductAffineOperator(a, l, U3, V3; op_type=:stiffness, quad_order=2)
A3 = get_matrix(op3)
b3 = get_vector(op3)

println("3D system size:  ", size(A3))   # (7*7*5, 7*7*5) = (245, 245)
println("3D symmetric:    ", maximum(abs, A3 - A3') < 1e-13)

---
## Connection map

```
CartesianDiscreteModel (×N 1D meshes)
       │
       ▼
 Measure(Ωk, q)  ──⊗──►  TensorProductMeasure            [Stage 1]
       │
       ▼
 TestFESpace / TrialFESpace  ──►  TensorProductFESpace     [Stage 2]
       │                          num_free_dofs = ∏ nf_k
       │
       ▼
 get_triangulation  ──►  TensorProductTriangulation        [Stage 3]
       │                   num_cells = ∏ nc_k
       │
       ▼
 assemble_subdomain_operators  ──►  SubdomainOperators{N}  [Stage 4a]
       │                             M, K, G, D, A per subdomain
       ▼
 assemble_stiffness_tensor (etc.)                          [Stage 4b]
       │   A = ∑_k kron(M_N,…,K_k,…,M_1)
       │
       ▼
 TensorProductAffineOperator → (A, b)                      [Stage 5]
       │
       ▼
       A \ b  →  u_dofs
```